# Retrospective CN3S monthly forecasts: 2024 and 2025

Compare **April and May issues** against **June–September observed monthly discharge**.
The workflow mirrors notebook 05: archived CDS forecast/hindcast ratios correct the
observed rainfall climatology, the median across centres drives the trained monthly
CN3S model, and its discharge is compared with observations.

**Interpretation:** the default calibration was trained through June 2026, including
these verification years. This is a retrospective diagnostic with a fixed fitted model,
not independent out-of-sample forecast skill. We do not retrain or overwrite that model.
Replace the calibration with one trained before the first issue for a held-out assessment.

Three rainfall scenarios use identical initial storage and antecedent rainfall:
- **Forecast:** corrected archived CDS rainfall (available at the selected issue).
- **Climatology:** a fixed pre-2024 rainfall climatology (baseline).
- **Actual rainfall:** what subsequently happened (hindsight diagnostic, unavailable at issue).

Forecast anchoring is disabled: no verification-month discharge is used to adjust forecasts.

In [ ]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

import json
from math import ceil, floor
from pathlib import Path

import cdsapi
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import xarray as xr
from IPython.display import display as show
from plotly.subplots import make_subplots

from cn3s import CN3S, CN3SParams
from hydro.forecast import load_forecast_grids
from hydro.retrospective import download_issue_forecasts
from hydro.station import Station

## Configuration and historical forecast systems

The issue date is the **11th**, after the multi-system release on the 10th; the nominal
CDS initialization is the first of that month. Only complete observed rainfall months
before initialization enter the model prefix. We do not assimilate partial issue-month
rainfall. This is deliberately conservative about what could have been known at issue.

The same **eight centres** are used for all cases. BOM is excluded because it joined
C3S in May 2025. As in notebook 05, only ECCC's GEM-NEMO component is used. Operational
system upgrades are retained: these are archived real forecasts, not today's systems
run retrospectively. The comparison therefore includes operational system changes.

Sources: [CDS monthly dataset](https://cds.climate.copernicus.eu/datasets/seasonal-monthly-single-levels)
and [ECMWF historical systems and release dates](https://confluence.ecmwf.int/spaces/CKB/pages/104239050/Summary+of+available+data).
Check that table before adding different years/issues.

In [ ]:
DATA_DIR = Path("/data/CN3S")
STATION_CODE = 29050000
STATION_DIR = DATA_DIR / "stations" / str(STATION_CODE)
OPTIMIZATION = "PlainNSE_alfa_livre.json"
CACHE_DIR = DATA_DIR / "forecasts" / "retrospective"
RESULTS_DIR = STATION_DIR / "retrospective" / Path(OPTIMIZATION).stem
DOWNLOAD_MISSING = True
CLIMATOLOGY_START = "2000-01-01"
CLIMATOLOGY_END = "2023-12-01"
LEADTIME_MONTHS = [1, 2, 3, 4, 5, 6]  # Lead 1 is the initialization month itself.
TARGET_MONTHS = [6, 7, 8, 9]
MIN_MONTHLY_COVERAGE = 1.0  # Score only fully observed months by default.

SYSTEMS_2024 = {
    "ukmo": "603",
    "ecmwf": "51",
    "meteo_france": "8",
    "dwd": "21",
    "cmcc": "35",
    "ncep": "2",
    "jma": "3",
    "eccc": "3",
}
SYSTEMS_2025_APR = {
    "ukmo": "604",
    "ecmwf": "51",
    "meteo_france": "8",
    "dwd": "22",
    "cmcc": "35",
    "ncep": "2",
    "jma": "3",
    "eccc": "5",
}
SYSTEMS_2025_MAY = {**SYSTEMS_2025_APR, "meteo_france": "9"}
ISSUES = {
    "2024-04-11": SYSTEMS_2024,
    "2024-05-11": SYSTEMS_2024,
    "2025-04-11": SYSTEMS_2025_APR,
    "2025-05-11": SYSTEMS_2025_MAY,
}
SCENARIO_LABELS = {
    "forecast": "CDS forecast rain",
    "climatology": "Climatological rain",
    "actual_rain": "Actual rain (hindsight)",
}
SCENARIO_COLORS = {"forecast": "red", "climatology": "purple", "actual_rain": "seagreen"}

## Preflight: local observations, calibration, and CDS credentials

Use saved observations from notebooks 01/02; Microsoft SQL sign-in is not required.
The local daily discharge snapshot is aggregated to calendar-month means in m³/s.
Coverage is the fraction of days with data; missing days are never filled. Monthly
rainfall must be continuous because gaps would alter the reservoir state and antecedent
rainfall sequence. These are the latest saved observations, not historical data vintages.

The baseline uses only 2000–2023 observations, so the 2024/2025 outcomes do not enter
rainfall bias correction or the historical discharge band. Calibration overlap is
reported separately and retained in exported results.

In [ ]:
station = Station.from_folder(STATION_DIR)
calibration_path = STATION_DIR / "optim" / OPTIMIZATION
calibration = json.loads(calibration_path.read_text())
params = CN3SParams.from_file(calibration_path)
training_cutoff = pd.to_datetime(calibration.get("metadata", {}).get("train_cutoff"))
if calibration.get("metadata", {}).get("frequency") != "M":
    msg = "Choose parameters calibrated at monthly frequency."
    raise ValueError(msg)

rain = pd.read_parquet(STATION_DIR / "Monthly" / "rain.parquet")["prec_mm"].sort_index()
daily_discharge = pd.read_parquet(STATION_DIR / "Daily" / "discharge.parquet")["q_m3s"]
observed = daily_discharge.sort_index().resample("MS").agg(["mean", "count"])
observed["coverage"] = observed["count"] / observed.index.days_in_month
rain_calendar = pd.date_range(rain.index.min(), rain.index.max(), freq="MS")
if not rain.index.equals(rain_calendar) or rain.isna().any():
    msg = "The saved monthly rainfall must be continuous and contain no missing values."
    raise ValueError(msg)
if pd.Timestamp(CLIMATOLOGY_END) >= pd.Timestamp(min(ISSUES)).to_period("M").to_timestamp():
    msg = "Climatology must end before the first initialization month."
    raise ValueError(msg)
climate_rain = rain.loc[CLIMATOLOGY_START:CLIMATOLOGY_END]
if len(climate_rain) != len(pd.date_range(CLIMATOLOGY_START, CLIMATOLOGY_END, freq="MS")):
    msg = "The rainfall snapshot does not cover the requested climatology period."
    raise ValueError(msg)
rain_climatology = climate_rain.groupby(climate_rain.index.month).mean()
climate_q = observed.loc[CLIMATOLOGY_START:CLIMATOLOGY_END]
climate_q = climate_q.loc[climate_q["coverage"] >= MIN_MONTHLY_COVERAGE, "mean"]
q_groups = climate_q.groupby(climate_q.index.month)
q_climatology = pd.DataFrame(
    {
        "median": q_groups.median(),
        "p10": q_groups.quantile(0.10),
        "p90": q_groups.quantile(0.90),
        "n_months": q_groups.count(),
    },
)
west, south, east, north = station.upstream_watershed.to_crs("EPSG:4326").total_bounds
AREA = [
    float(ceil(north) + 1),
    float(floor(west) - 1),
    float(floor(south) - 1),
    float(ceil(east) + 1),
]
if DOWNLOAD_MISSING:
    cds_client = cdsapi.Client(quiet=True)  # Fail early if ~/.cdsapirc is not configured.

issue_metadata = pd.DataFrame(
    [
        {
            "issue_date": pd.Timestamp(issue),
            "initialization": pd.Timestamp(issue).to_period("M").to_timestamp(),
            "training_cutoff": training_cutoff,
            "calibration_overlaps_issue": pd.isna(training_cutoff)
            or training_cutoff >= pd.Timestamp(issue),
            "n_centres": len(systems),
            "systems": json.dumps(systems, sort_keys=True),
        }
        for issue, systems in ISSUES.items()
    ],
)
show(issue_metadata)
print(f"Station: {station.name}; ACT: {params.act} calendar month(s); CDS area: {AREA}")
if issue_metadata["calibration_overlaps_issue"].any():
    print("RETROSPECTIVE DIAGNOSTIC: calibration overlaps issues or its cutoff is unknown.")
show(q_climatology)
show(
    observed.loc[observed.index.year.isin([2024, 2025]) & observed.index.month.isin(TARGET_MONTHS)],
)

## Download archived issue data (resumable)

For each issue download six leads of **ensemble mean precipitation** and the **matched
hindcast climate mean**. Six leads include April–September for an April initialization,
and May–October for May. This also supplies the issue-month forcing needed to propagate
CN3S storage before June. ACT is handled later by the model, not by relabeling CDS leads.

Only the basin bounding box plus a one-degree margin is requested. Each completed file
is cached with its exact request and reused on reruns. With eight centres and four
issues, a fresh run makes 64 small CDS requests; CDS queueing can take time. Interrupting
and rerunning resumes from completed files. Existing notebook 03/05 caches are separate.

Set `DOWNLOAD_MISSING=False` to analyze a previously downloaded cache. If credentials
or dataset access fail, configure CDS access using the dataset's API instructions;
this notebook does not accept terms or install dependencies automatically.

In [ ]:
download_manifests = []
for issue_text, systems in ISSUES.items():
    initialization = pd.Timestamp(issue_text).to_period("M").to_timestamp()
    issue_cache = CACHE_DIR / f"{initialization:%Y%m}"
    manifest = download_issue_forecasts(
        issue_cache,
        initialization,
        systems,
        leads=LEADTIME_MONTHS,
        area=AREA,
        download_missing=DOWNLOAD_MISSING,
    )
    manifest["issue_date"] = issue_text
    download_manifests.append(manifest)
download_manifest = pd.concat(download_manifests, ignore_index=True)
show(download_manifest)

## Reconstruct each forecast and run the three scenarios

The correction retains notebook 05's **basin mean of grid-cell forecast/hindcast ratios**,
then multiplies the observed monthly rainfall climatology. The rainfall climatology is
now explicitly fixed before the verification years. The median across centres is a
rainfall summary, not a median discharge forecast or a probability interval.

All scenarios share the complete rainfall prefix up to the month before initialization,
which reconstructs CN3S storage using the calibrated `r0`. No actual rainfall from the
initialization month onward enters the forecast or climatology scenario. Actual future
rainfall is used only in the explicitly labeled hindsight scenario.

CN3S already shifts discharge timestamps by `ACT` calendar months. A June discharge
prediction with ACT=1 is driven by May rainfall, so its CDS rainfall lead differs from
its discharge lead. Both labels are retained in the comparison table. No 15-day plotting
shift is used here.

In [ ]:
issue_results = {}
rainfall_results = {}
comparison_rows = []
for issue_text, systems in ISSUES.items():
    issue_date = pd.Timestamp(issue_text)
    initialization = issue_date.to_period("M").to_timestamp()
    grids = load_forecast_grids(
        CACHE_DIR / f"{initialization:%Y%m}",
        initialization,
        list(systems),
        station.upstream_watershed,
    )
    forecast_months = pd.DatetimeIndex(grids.time.values)
    expected_months = pd.DatetimeIndex(
        [initialization + pd.DateOffset(months=lead - 1) for lead in LEADTIME_MONTHS],
    )
    if not forecast_months.equals(expected_months):
        msg = f"CDS leads do not match the requested calendar for {issue_text}."
        raise ValueError(msg)
    # Reject an undefined multiplicative correction instead of hiding zero denominators.
    if ((grids["hindcast"] <= 0) & grids["prec"].notnull()).any().item():  # noqa: PD004 -- xarray API
        msg = f"Nonpositive hindcast rainfall in the basin for {issue_text}."
        raise ValueError(msg)
    ratio = (grids["prec"] / grids["hindcast"]).mean(dim=["latitude", "longitude"])
    if not np.isfinite(ratio.values).all():
        msg = f"Missing or nonfinite forecast ratios for {issue_text}."
        raise ValueError(msg)
    reference_rain = xr.DataArray(
        rain_climatology.reindex(forecast_months.month).to_numpy(),
        dims="time",
        coords={"time": forecast_months},
    )
    corrected_rain = ratio * reference_rain
    future_rain = pd.DataFrame(
        {
            "forecast": corrected_rain.median("model").to_pandas(),
            "climatology": reference_rain.to_pandas(),
            "actual_rain": rain.reindex(forecast_months),
        },
    )
    history = rain.loc[rain.index < initialization]
    rain_scenarios = pd.concat(
        [
            pd.DataFrame(dict.fromkeys(SCENARIO_LABELS, history)),
            future_rain,
        ],
    )
    expected_inputs = pd.date_range(history.index.min(), forecast_months.max(), freq="MS")
    if not rain_scenarios.index.equals(expected_inputs) or rain_scenarios.isna().any().any():
        msg = f"Incomplete monthly rainfall inputs for {issue_text}."
        raise ValueError(msg)
    predictions = {}
    for scenario in SCENARIO_LABELS:
        model = CN3S(params, freq="M")
        model.run(rain_scenarios[scenario], pbar=False)
        predictions[scenario] = model.results["q_m3s"].astype(float)
    discharge = pd.DataFrame(predictions)
    targets = pd.DatetimeIndex([pd.Timestamp(issue_date.year, month, 1) for month in TARGET_MONTHS])
    if (targets <= initialization).any() or discharge.reindex(targets).isna().any().any():
        msg = f"Requested target months must be after issue and covered by {issue_text}."
        raise ValueError(msg)
    case = discharge.reindex(targets)
    case["observed"] = observed["mean"].reindex(targets)
    case["coverage"] = observed["coverage"].reindex(targets)
    issue_results[issue_text] = case
    rainfall_results[issue_text] = future_rain
    for scenario in SCENARIO_LABELS:
        rows = case[[scenario, "observed", "coverage"]].rename(columns={scenario: "predicted"})
        rows = rows.rename_axis("target_month").reset_index()
        rows["issue_date"] = issue_date
        rows["scenario"] = scenario
        rows["discharge_lead_months"] = (
            (rows["target_month"].dt.year - initialization.year) * 12
            + rows["target_month"].dt.month
            - initialization.month
        )
        rows["rainfall_source_month"] = rows["target_month"].map(
            lambda date: date - pd.DateOffset(months=params.act),
        )
        rows["cds_rain_lead"] = rows["discharge_lead_months"] - params.act + 1
        rows["calibration_overlaps_issue"] = (
            pd.isna(training_cutoff) or training_cutoff >= issue_date
        )
        comparison_rows.append(rows)
comparison = pd.concat(comparison_rows, ignore_index=True)
comparison["error"] = comparison["predicted"] - comparison["observed"]
comparison["eligible"] = (
    comparison["coverage"].ge(MIN_MONTHLY_COVERAGE) & comparison["observed"].notna()
)
show(comparison)

## Monthly discharge comparison by issue

Black markers show what happened; red is the archived forecast-rain scenario, purple
is the climatological-rain baseline, and green is the actual-rain hindsight diagnostic.
The gray P10–P90 band and dashed gray median describe **pre-2024 monthly discharge**,
not forecast uncertainty. Titles identify the issue, while x positions identify the
verification month. A difference remaining with actual rainfall indicates limitations
of the fixed hydrological model/state rather than only rainfall-forecast error.

In [ ]:
discharge_figure = make_subplots(
    rows=ceil(len(ISSUES) / 2),
    cols=2,
    subplot_titles=[f"Issue {pd.Timestamp(issue):%d/%m/%Y}" for issue in ISSUES],
)
for panel, case in enumerate(issue_results.values()):
    row, col = panel // 2 + 1, panel % 2 + 1
    reference = q_climatology.reindex(case.index.month)
    discharge_figure.add_trace(
        go.Scatter(
            x=case.index,
            y=reference["p10"],
            mode="lines",
            line={"width": 0},
            showlegend=False,
            hoverinfo="skip",
        ),
        row=row,
        col=col,
    )
    discharge_figure.add_trace(
        go.Scatter(
            x=case.index,
            y=reference["p90"],
            mode="lines",
            line={"width": 0},
            fill="tonexty",
            fillcolor="rgba(128,128,128,0.18)",
            name="Historical monthly P10-P90",
            legendgroup="band",
            showlegend=panel == 0,
            hoverinfo="skip",
        ),
        row=row,
        col=col,
    )
    discharge_figure.add_trace(
        go.Scatter(
            x=case.index,
            y=reference["median"],
            mode="lines",
            line={"color": "gray", "dash": "dash", "width": 1},
            name="Historical monthly median",
            legendgroup="median",
            showlegend=panel == 0,
        ),
        row=row,
        col=col,
    )
    discharge_figure.add_trace(
        go.Scatter(
            x=case.index,
            y=case["observed"],
            mode="lines+markers",
            line={"color": "black", "width": 3},
            name="Observed",
            legendgroup="observed",
            customdata=case["coverage"],
            showlegend=panel == 0,
            hovertemplate=(
                "%{x|%b %Y}: %{y:,.0f} m³/s<br>Coverage: %{customdata:.0%}<extra>Observed</extra>"
            ),
        ),
        row=row,
        col=col,
    )
    for scenario, label in SCENARIO_LABELS.items():
        discharge_figure.add_trace(
            go.Scatter(
                x=case.index,
                y=case[scenario],
                mode="lines+markers",
                line={"color": SCENARIO_COLORS[scenario], "dash": "dash"},
                name=label,
                legendgroup=scenario,
                showlegend=panel == 0,
                hovertemplate="%{x|%b %Y}: %{y:,.0f} m³/s<extra>%{fullData.name}</extra>",
            ),
            row=row,
            col=col,
        )
    discharge_figure.update_xaxes(
        tickmode="array",
        tickvals=case.index,
        tickformat="%b %Y",
        row=row,
        col=col,
    )
discharge_figure.update_yaxes(title_text="Monthly mean discharge (m³/s)", rangemode="tozero")
discharge_figure.update_layout(
    title="Retrospective CN3S comparison — fixed calibration",
    template="plotly_white",
    height=350 * ceil(len(ISSUES) / 2) + 150,
    legend={"orientation": "h", "y": -0.18},
    hovermode="x unified",
)
discharge_figure.show()

## Rainfall forecast versus actual rainfall

All six forcing months are shown, including the initialization month. These are
calendar-month totals in mm. The model's ACT lag shifts their discharge response later.
This plot helps distinguish rainfall error from hydrological-model error.

In [ ]:
rainfall_figure = make_subplots(
    rows=ceil(len(ISSUES) / 2),
    cols=2,
    subplot_titles=[f"Initialization {pd.Timestamp(issue):%m/%Y}" for issue in ISSUES],
)
for panel, frame in enumerate(rainfall_results.values()):
    row, col = panel // 2 + 1, panel % 2 + 1
    rainfall_figure.add_trace(
        go.Bar(
            x=frame.index,
            y=frame["actual_rain"],
            name="Actual rainfall",
            marker_color="steelblue",
            legendgroup="actual",
            showlegend=panel == 0,
        ),
        row=row,
        col=col,
    )
    for scenario in ("forecast", "climatology"):
        rainfall_figure.add_trace(
            go.Scatter(
                x=frame.index,
                y=frame[scenario],
                mode="lines+markers",
                line={"color": SCENARIO_COLORS[scenario]},
                name=SCENARIO_LABELS[scenario],
                legendgroup=scenario,
                showlegend=panel == 0,
            ),
            row=row,
            col=col,
        )
    rainfall_figure.update_xaxes(
        tickmode="array",
        tickvals=frame.index,
        tickformat="%b",
        row=row,
        col=col,
    )
rainfall_figure.update_yaxes(title_text="Monthly rainfall (mm)", rangemode="tozero")
rainfall_figure.update_layout(
    template="plotly_white",
    height=350 * ceil(len(ISSUES) / 2) + 120,
    legend={"orientation": "h", "y": -0.18},
    hovermode="x unified",
)
rainfall_figure.show()

## Errors by issue and forecast lead

MAE, RMSE, and mean bias are in m³/s; positive bias means overprediction. Only months
meeting the configured observation coverage are scored, on the same observed support
for all scenarios. The sample counts are small (four targets per issue), so these are
descriptive comparisons rather than a statistically established forecast-skill estimate.

A positive RMSE improvement versus climatological rainfall means the archived CDS
rainfall improved this hydrological forecast on the selected months. Calibration overlap
still applies. The actual-rain case is a hindsight diagnostic and is never counted as
an operational forecast. Lead summaries pool issue–target pairs; the same observed
month can occur at different leads from different issues.

In [ ]:
def error_metrics(frame: pd.DataFrame) -> pd.Series:
    """
    Summarize paired monthly discharge errors.

    Args:
        frame: Predicted and observed monthly discharge in m³/s on common support.

    Returns:
        Number of paired months, MAE, RMSE and mean bias in m³/s.

    """
    errors = frame["predicted"] - frame["observed"]
    return pd.Series(
        {
            "n": errors.count(),
            "MAE": errors.abs().mean(),
            "RMSE": np.sqrt(errors.pow(2).mean()),
            "bias": errors.mean(),
        },
    )


eligible = comparison.loc[comparison["eligible"]]
if eligible.empty:
    msg = "No verification months meet MIN_MONTHLY_COVERAGE."
    raise ValueError(msg)
scores = eligible.groupby(["issue_date", "scenario"])[["predicted", "observed"]].apply(
    error_metrics,
)
lead_scores = eligible.groupby(["discharge_lead_months", "scenario"])[
    ["predicted", "observed"]
].apply(error_metrics)
rmse_by_issue = scores.reset_index().pivot_table(
    index="issue_date",
    columns="scenario",
    values="RMSE",
)
rmse_by_issue["forecast_improvement_vs_climatology_pct"] = 100 * (
    1 - rmse_by_issue["forecast"] / rmse_by_issue["climatology"].replace(0, np.nan)
)
show(scores.round(2))
show(rmse_by_issue.round(2))
show(lead_scores.round(2))

## Save reproducible results

Exports are separate from calibration and preparation files. The manifest records the
exact issue/system choices, observation snapshot paths, climatology period, ACT, and
training overlap. Re-running overwrites these analysis outputs; the CDS cache is reused.
Interactive HTML figures include Plotly so they can be opened without a Python kernel.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(RESULTS_DIR / "monthly_comparison.csv", index=False)
scores.to_csv(RESULTS_DIR / "scores_by_issue.csv")
lead_scores.to_csv(RESULTS_DIR / "scores_by_lead.csv")
issue_metadata.to_csv(RESULTS_DIR / "issues.csv", index=False)
for issue_text, frame in rainfall_results.items():
    frame.to_csv(RESULTS_DIR / f"rainfall_{pd.Timestamp(issue_text):%Y%m%d}.csv")
manifest = {
    "station": STATION_CODE,
    "calibration": str(calibration_path),
    "training_cutoff": str(training_cutoff),
    "calibrated_params": calibration["params"],
    "act_months": params.act,
    "assessment": "retrospective fixed-calibration diagnostic",
    "issues": ISSUES,
    "target_months": TARGET_MONTHS,
    "cds_leads": LEADTIME_MONTHS,
    "climatology_period": [CLIMATOLOGY_START, CLIMATOLOGY_END],
    "minimum_monthly_coverage": MIN_MONTHLY_COVERAGE,
    "area": AREA,
    "rain_snapshot": str(STATION_DIR / "Monthly" / "rain.parquet"),
    "discharge_snapshot": str(STATION_DIR / "Daily" / "discharge.parquet"),
    "generated_at": pd.Timestamp.now(tz="America/Sao_Paulo").isoformat(),
}
(RESULTS_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
discharge_figure.write_html(RESULTS_DIR / "discharge_comparison.html", include_plotlyjs=True)
rainfall_figure.write_html(RESULTS_DIR / "rainfall_comparison.html", include_plotlyjs=True)
print(f"Saved comparison tables and figures to {RESULTS_DIR}")